# 04 — Cara Kerja Aplikasi Streamlit

**Kelompok 12 · Kelas B2 · Prediksi Customer Churn Telco**

Aplikasi web tetap berupa file `.py` karena Streamlit Community Cloud hanya dapat menjalankan file Python.
Notebook ini menjalankan fungsi inti aplikasi langkah demi langkah dengan model yang sama, sehingga alur
prediksi dapat dijelaskan tanpa membuka web.

| File | Isi |
|---|---|
| `app.py` | Halaman Streamlit: memuat model, tab Prediksi Single, Prediksi Batch (CSV), Dashboard & Model, EDA & Dokumentasi |
| `ui.py` | Tampilan: tema Terang/Gelap, kartu, gauge risiko, tabel confusion matrix |
| `prediction.py` | Validasi input, perhitungan peluang churn, dan tingkat risiko |
| `training_data.py` | Pembagian data latih/uji dan rumus metrik evaluasi (dipakai notebook, aplikasi, dan tes) |
| `tests/test_app.py` | 7 pengujian otomatis untuk fungsi prediksi dan tampilan aplikasi |

## 1. Import library

Fungsi yang diimpor dari `prediction.py` adalah fungsi yang sama dengan yang dipanggil `app.py`.

In [1]:
import hashlib
import json

import joblib
import pandas as pd
from IPython.display import display

from prediction import (
    InputValidationError, example_customers, input_notes, input_schema,
    predict_customers, risk_levels, validate_input,
)
from training_data import BASE

## 2. Memuat model dan memeriksa keaslian file

Seperti `load_models()` di `app.py`: hash SHA-256 `model_churn.pkl` dan `calibrator.pkl` dicocokkan dengan
`deployment_info.json` sebelum model dipakai. Jika berbeda, aplikasi berhenti dan meminta model dilatih ulang.

In [2]:
deployment = json.loads((BASE / "deployment_info.json").read_text(encoding="utf-8"))
for filename, key in [("model_churn.pkl", "model_sha256"), ("calibrator.pkl", "calibrator_sha256")]:
    digest = hashlib.sha256((BASE / filename).read_bytes()).hexdigest()
    status = "cocok" if digest == deployment[key] else "TIDAK COCOK"
    print(f"{filename}: {status}")

pipe = joblib.load(BASE / "model_churn.pkl")
cal = joblib.load(BASE / "calibrator.pkl")
threshold = float(deployment["threshold"])
print(f"Threshold keputusan: {threshold} | batas risiko tinggi: {deployment['risk_high_boundary']}")

model_churn.pkl: cocok
calibrator.pkl: cocok
Threshold keputusan: 0.32 | batas risiko tinggi: 0.6


## 3. Skema input dari model

`input_schema` membaca daftar 18 kolom dan kategori yang valid langsung dari pipeline terlatih. Pilihan pada
form aplikasi dan validasi CSV memakai skema ini.

In [3]:
schema = input_schema(pipe)
print(f"Jumlah kolom input: {len(schema['columns'])}")
display(pd.Series({col: ", ".join(values) for col, values in schema["categories"].items()},
                  name="Kategori valid").to_frame())

Jumlah kolom input: 18


,Kategori valid
gender,"Female, Male"
Partner,"No, Yes"
Dependents,"No, Yes"
PhoneService,"No, Yes"
MultipleLines,"No, No phone service, Yes"
InternetService,"DSL, Fiber optic, No"
OnlineSecurity,"No, No internet service, Yes"
OnlineBackup,"No, No internet service, Yes"
DeviceProtection,"No, No internet service, Yes"
TechSupport,"No, No internet service, Yes"


## 4. Contoh pelanggan

Dua pelanggan contoh dari `example_customers()`, sama dengan isi template CSV yang bisa diunduh di aplikasi:
pelanggan baru berkontrak bulanan dengan Fiber optic, dan pelanggan lama berkontrak dua tahun dengan DSL.

In [4]:
customers = example_customers()
display(customers.T)

,0,1
gender,Male,Female
SeniorCitizen,0,0
Partner,No,Yes
Dependents,No,Yes
tenure,2,60
PhoneService,Yes,Yes
MultipleLines,No,Yes
InternetService,Fiber optic,DSL
OnlineSecurity,No,Yes
OnlineBackup,No,Yes


## 5. Validasi input

`validate_input` memeriksa kolom wajib, rentang angka (`tenure` 0–72 bulan, `MonthlyCharges` 0–120 dolar,
`SeniorCitizen` 0/1), kategori yang valid, dan kecocokan antar-layanan. Data yang lolos dikembalikan dalam
urutan kolom yang dipakai model.

In [5]:
clean = validate_input(customers, schema)
print(f"Lolos validasi: {len(clean)} pelanggan, {clean.shape[1]} kolom")

Lolos validasi: 2 pelanggan, 18 kolom


## 6. Peluang churn dan tingkat risiko

`predict_customers` menghitung peluang churn dari kalibrator. Pelanggan diprediksi CHURN bila peluangnya
≥ threshold. `risk_levels` membagi peluang menjadi RENDAH (< threshold), SEDANG, dan TINGGI (≥ batas risiko
tinggi).

In [6]:
probabilities, predictions = predict_customers(customers, cal, schema, threshold)
display(pd.DataFrame({
    "Peluang churn (%)": (probabilities * 100).round(2),
    "Prediksi": ["CHURN" if p else "TETAP" for p in predictions],
    "Tingkat risiko": risk_levels(probabilities, threshold),
}))

,Peluang churn (%),Prediksi,Tingkat risiko
0,74.79,CHURN,TINGGI
1,0.99,TETAP,RENDAH


## 7. Catatan untuk input di luar data latih

`input_notes` memberi peringatan, tanpa menolak input, bila `tenure = 0` atau tagihan berada di luar rentang
data latih ($18,25–$118,75).

In [7]:
unusual = customers.assign(tenure=0, MonthlyCharges=119.5)
for note in input_notes(unusual):
    print("-", note)

- Ada pelanggan dengan lama berlangganan 0 bulan. Model dilatih pada pelanggan yang sudah berlangganan minimal 1 bulan, jadi hasil untuk pelanggan baru ini belum teruji.
- Ada tagihan di luar rentang data training ($18,25–$118,75). Prediksi tetap tersedia, tetapi model belum diuji pada nilai tagihan tersebut.


## 8. Contoh input yang ditolak

Input yang tidak valid menghasilkan `InputValidationError` dengan pesan yang menyebutkan bagian yang harus
diperbaiki. Pesan inilah yang ditampilkan aplikasi; prediksi tidak dijalankan.

In [8]:
invalid_inputs = {
    "Kolom tenure tidak ada": customers.drop(columns="tenure"),
    "Kontrak tidak dikenal": customers.assign(Contract="Two years"),
    "Tanpa internet tetapi memakai OnlineSecurity": customers.assign(InternetService="No"),
}
for title, frame in invalid_inputs.items():
    try:
        predict_customers(frame, cal, schema, threshold)
    except InputValidationError as error:
        print(f"{title}:\n{error}\n")

Kolom tenure tidak ada:
Kolom wajib yang belum ada: tenure.

Kontrak tidak dikenal:
Contract hanya menerima: Month-to-month, One year, Two year (baris data 1, 2).

Tanpa internet tetapi memakai OnlineSecurity:
InternetService=No mengharuskan OnlineSecurity=No internet service (baris data 1, 2).
InternetService=No mengharuskan OnlineBackup=No internet service (baris data 1, 2).
InternetService=No mengharuskan DeviceProtection=No internet service (baris data 1, 2).
InternetService=No mengharuskan TechSupport=No internet service (baris data 1, 2).
InternetService=No mengharuskan StreamingTV=No internet service (baris data 1, 2).
InternetService=No mengharuskan StreamingMovies=No internet service (baris data 1, 2).



## 9. Prediksi banyak pelanggan sekaligus (batch CSV)

Seperti tab Prediksi Batch: seluruh dataset mentah dibaca dari CSV, kolom tambahan seperti `customerID` dan
`Churn` diabaikan, lalu setiap pelanggan diberi peluang dan tingkat risiko.

In [9]:
batch = pd.read_csv(BASE / "Telco-Customer-Churn.csv")
batch_prob, batch_pred = predict_customers(batch, cal, schema, threshold)
result = batch.assign(Peluang_churn=batch_prob.round(4), Prediksi=batch_pred,
                      Risiko=risk_levels(batch_prob, threshold))
print(f"{len(result):,} pelanggan diproses, {int(batch_pred.sum()):,} diprediksi churn")
display(result["Risiko"].value_counts().reindex(["RENDAH", "SEDANG", "TINGGI"]).to_frame("Jumlah pelanggan"))
display(result[["customerID", "Contract", "tenure", "MonthlyCharges", "Peluang_churn", "Risiko"]].head())

7,043 pelanggan diproses, 2,532 diprediksi churn


,Jumlah pelanggan
Risiko,
RENDAH,4511
SEDANG,1506
TINGGI,1026


,customerID,Contract,tenure,MonthlyCharges,Peluang_churn,Risiko
0,7590-VHVEG,Month-to-month,1,29.85,0.5379,SEDANG
1,5575-GNVDE,One year,34,56.95,0.0527,RENDAH
2,3668-QPYBK,Month-to-month,2,53.85,0.2802,RENDAH
3,7795-CFOCW,One year,45,42.30,0.0396,RENDAH
4,9237-HQITU,Month-to-month,2,70.70,0.6628,TINGGI


## 10. Menjalankan aplikasi dan pengujian

Dari folder utama repository:

```bash
streamlit run app.py
python -m unittest discover -s tests -v
```